In [1]:
pip install contextily

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install matplotlib-scalebar

Note: you may need to restart the kernel to use updated packages.


In [2]:
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import contextily as ctx
from pyproj import Transformer
from datetime import datetime

# === File path shapefile ===
kk_fp = r"D:\04_Data Shapefile\Data Kawasan Konservasi\2024_Kawasan Konservasi\KK_Reg117_2024_WGS84_fix.shp"
hotspot_fp = r"D:\04_Data Shapefile\Data Kebakaran\01_Hotspot (Monthly)\HS_2026\2026_07_21\HS_20260720_20260721_KK.shp" # GANTI TANGGAL

# === Load shapefiles ===
try:
    kk = gpd.read_file(kk_fp)
    hotspot = gpd.read_file(hotspot_fp)
except Exception as e:
    raise Exception(f"Gagal memuat shapefile: {str(e)}")

# === Filter hanya hotspot dengan confidence High ===
hotspot = hotspot[hotspot['CONFIDENCE'] == 'High']

# === CRS adjustment ===
if kk.crs != hotspot.crs:
    kk = kk.to_crs(hotspot.crs)

# === Reproject ke Web Mercator ===
kk_web = kk.to_crs(epsg=3857)
hotspot_web = hotspot.to_crs(epsg=3857)

# === Output folder ===
peta_folder = os.path.join(os.path.dirname(hotspot_fp), "Peta_High")
os.makedirs(peta_folder, exist_ok=True)

# === Fungsi ubah koordinat ke derajat menit desimal ===
def decimal_degrees_to_dm(val, axis='lon'):
    is_positive = val >= 0
    val = abs(val)
    degrees = int(val)
    minutes = (val - degrees) * 60
    hemi = {'lon': ('E', 'W'), 'lat': ('N', 'S')}
    suffix = hemi[axis][0] if is_positive else hemi[axis][1]
    return f"{degrees}°{minutes:.2f}' {suffix}"

# === Fungsi format tanggal ===
def format_tanggal_bahasa(date):
    bulan = {
        1: "Januari", 2: "Februari", 3: "Maret", 4: "April",
        5: "Mei", 6: "Juni", 7: "Juli", 8: "Agustus",
        9: "September", 10: "Oktober", 11: "November", 12: "Desember"
    }
    return f"{date.day} {bulan[date.month]} {date.year}"

# === Loop tiap NKWS ===
nkws_list = hotspot_web['NKWS'].dropna().unique()

for nkws in nkws_list:
    try:
        # Bersihkan nama file dari karakter khusus
        nkws_clean = nkws.replace(" ", "_").replace("/", "-").replace("\\", "-")
        
        subset_hotspot = hotspot_web[hotspot_web['NKWS'] == nkws]
        subset_kk = kk_web[kk_web['NKWS'] == nkws]

        if subset_hotspot.empty or subset_kk.empty:
            print(f"❌ Lewati '{nkws}' karena tidak ada data lengkap.")
            continue

        # Ambil tanggal dari field TANGGAL
        tanggal_awal = subset_hotspot['TANGGAL'].min()
        tanggal_akhir = subset_hotspot['TANGGAL'].max()
        if tanggal_awal.date() == tanggal_akhir.date():
            tanggal_str = format_tanggal_bahasa(tanggal_awal)
        else:
            tanggal_str = f"{format_tanggal_bahasa(tanggal_awal)} - {format_tanggal_bahasa(tanggal_akhir)}"

        # Nama file yang aman
        output_fp = os.path.normpath(
            os.path.join(peta_folder, f"Peta_HS_High_{nkws_clean}.jpg")
        )

        # Plotting
        fig, ax = plt.subplots(figsize=(10, 10))
        subset_hotspot.plot(
            ax=ax, 
            color='red', 
            markersize=70, 
            edgecolor='black', 
            marker='*',
            label='Hotspot High'
        )
        subset_kk.boundary.plot(ax=ax, edgecolor='fuchsia', linewidth=1, label='Kawasan Konservasi')

        # Basemap citra satelit
        try:
            ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs=subset_kk.crs)
        except Exception as e:
            print(f"⚠️ Basemap gagal: {e}")

        # Zoom otomatis
        bounds = subset_kk.total_bounds
        margin_x = (bounds[2] - bounds[0]) * 0.05
        margin_y = (bounds[3] - bounds[1]) * 0.05
        ax.set_xlim(bounds[0] - margin_x, bounds[2] + margin_x)
        ax.set_ylim(bounds[1] - margin_y, bounds[3] + margin_y)

        # Format ticks koordinat
        transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
        xticks = ax.get_xticks()
        yticks = ax.get_yticks()
        ax.set_xticklabels(
            [decimal_degrees_to_dm(transformer.transform(x, bounds[1])[0], 'lon') 
             for x in xticks], fontsize=6
        )
        ax.set_yticklabels(
            [decimal_degrees_to_dm(transformer.transform(bounds[0], y)[1], 'lat') 
             for y in yticks], fontsize=6
        )

        # Judul dan legenda
        ax.set_title(f"Peta Hotspot High - {nkws}\nTanggal: {tanggal_str}", fontsize=14)
        ax.legend(loc='upper right')
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        # Simpan peta
        plt.tight_layout()
        plt.savefig(output_fp, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✅ Peta High untuk '{nkws}' disimpan:\n{output_fp}")

    except Exception as e:
        print(f"❌ Error saat memproses '{nkws}': {str(e)}")
        plt.close()  # Pastikan figure ditutup jika ada error

print("\n✨ Proses selesai!")

C:\Users\KK EDEN\AppData\Local\Temp\ipykernel_63232\2435630009.py:110: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
C:\Users\KK EDEN\AppData\Local\Temp\ipykernel_63232\2435630009.py:114: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(


✅ Peta High untuk 'TN Way Kambas' disimpan:
D:\04_Data Shapefile\Data Kebakaran\01_Hotspot (Monthly)\HS_2026\2026_07_21\Peta_High\Peta_HS_High_TN_Way_Kambas.jpg


C:\Users\KK EDEN\AppData\Local\Temp\ipykernel_63232\2435630009.py:110: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
C:\Users\KK EDEN\AppData\Local\Temp\ipykernel_63232\2435630009.py:114: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(


✅ Peta High untuk 'TN Gandang Dewata' disimpan:
D:\04_Data Shapefile\Data Kebakaran\01_Hotspot (Monthly)\HS_2026\2026_07_21\Peta_High\Peta_HS_High_TN_Gandang_Dewata.jpg

✨ Proses selesai!
